In [ ]:
from shapely.geometry import LineString, Polygon, Point
import geojson
from geojson import Feature, FeatureCollection
import pandas as pd
import geopandas as gpd
import numpy as np
import osmnx as ox
import os
import networkx as nx
from IPython.display import IFrame

In [ ]:
os.chdir("C:/...")

In [ ]:
city_transport = pd.read_excel("marshruty-dvizheniya-gorodskogo-transporta.xlsx")

In [ ]:
city_transport[["Longitude", "Latitude"]] = city_transport["Координаты остановки"].str.split(",", expand = True)

In [ ]:
city_transport = gpd.GeoDataFrame(
    city_transport, geometry=gpd.points_from_xy(city_transport.Longitude,
                                                city_transport.Latitude))

In [ ]:
TERITORY_NAME = 'Петроградский район, Санкт-Петербург'

territory = ox.geocode_to_gdf(TERITORY_NAME)

In [ ]:
city_transport.set_crs(epsg = "4326", inplace = True)
transport_by_district = city_transport.sjoin(territory)

In [ ]:
m = territory.explore(tiles="CartoDB positron")
transport_by_district.explore(m=m, marker_kwds={"radius": 3, "color": "red"})

In [ ]:
import osmnx as ox
%matplotlib inline

In [ ]:
G = ox.graph_from_place(TERITORY_NAME, network_type="drive")
ox.plot_graph(ox.project_graph(G))

In [ ]:
m1 = ox.plot_graph_folium(G, popup_attribute="name", weight=2, color="#8b0000")

In [ ]:
import folium
mapit = None
latlon = [[float(x.split(',')[0]), float(x.split(',')[1])] for x in city_transport["Координаты остановки"].to_list()]

In [ ]:
nodes, edges = ox.utils_graph.graph_to_gdfs(G)
nodes.head()

In [ ]:
m = territory.explore(tiles="CartoDB positron")
m1 = transport_by_district.explore(m=m, marker_kwds={"radius": 3, "color": "red"})
edges.explore(m=m1, marker_kwds={ "color": "red"})

In [ ]:
transport_by_district[["Идентификатор маршрута",	"Числовой номер маршрута",	"Наименование маршрута"]]

In [ ]:
all_transport = set(transport_by_district["Идентификатор маршрута"].to_list())

In [ ]:
transport_dfs = [transport_by_district[transport_by_district["Идентификатор маршрута"] == tr].drop(columns=["index_right"]) 
                 for tr in all_transport if transport_by_district[transport_by_district["Идентификатор маршрута"] == tr]
                 ["Направление движения"].value_counts().shape[0] == 2]             

In [ ]:
m = territory.explore(tiles="CartoDB positron")
m1 = transport_dfs[0].explore(m=m, marker_kwds={"radius": 3, "color": "red"})
edges.explore(m=m1, marker_kwds={ "color": "red"})

In [ ]:
for tr in transport_dfs:
    tr.set_crs(4326, inplace=True)

In [ ]:
transport_dfs[0].columns

In [ ]:
edges.columns

In [ ]:
import geopandas as gpd
import shapely.wkt
import shapely.geometry

def get_routes(df, edges):
    offset = 10
    bbox = df.bounds + [-offset, -offset, offset, offset]
    # создаю полигон вокруг точки 
    
    hits = bbox.apply(lambda row: list(edges.sindex.intersection(row)), axis=1)
    # сколько раз пересекают полигон вокруг точки уличная сеть
    
    tmp = pd.DataFrame({
    # датафрейм с указанием количества пересечений
    "pt_idx": np.repeat(hits.index, hits.apply(len)),
    "line_i": np.concatenate(hits.values)
    })

    # Накладываю линии УДС с bbox
    tmp = tmp.join(edges.reset_index(drop=True), on="line_i")
    # соединяю с геометрией остановки
    tmp = tmp.join(df.geometry.rename("point"), on="pt_idx")
    # Превращаю в геодатафрейм
    tmp = gpd.GeoDataFrame(tmp, geometry="geometry", crs=df.crs)

    tmp["snap_dist"] = tmp.geometry.distance(gpd.GeoSeries(tmp.point))
    # Определяю расстояние от остановки до пересекающих полигон улиц

    # Выбираю и сортирую по расстоянию до улиц
    tmp = tmp.loc[tmp.snap_dist <= 1]
    tmp = tmp.sort_values(by=["snap_dist"])

    # Выбираю ближайшую линию для каждой остановки
    closest = tmp.groupby("pt_idx").first()
    # собираю датафрейм с остановками и ближайшими ребрами УДС
    closest = gpd.GeoDataFrame(closest, geometry="geometry")
    closest.set_crs(epsg = "4326", inplace = True)

    closest["Transport_number"] = df["Числовой номер маршрута"]

    return closest




In [ ]:
final_transport_df = get_routes(transport_dfs[0], edges)
for i in range(1, len(transport_dfs)):
    final_transport_df = gpd.GeoDataFrame(pd.concat([final_transport_df, get_routes(transport_dfs[i], edges)], ignore_index=True))


In [ ]:
m = territory.explore(tiles="CartoDB positron")
m1 = final_transport_df["point"].explore(m=m, marker_kwds={"radius": 3, "color": "red"})
final_transport_df["geometry"].explore(m=m1)

In [ ]:
final_transport_df.columns

In [ ]:
stops_df = final_transport_df[['line_i', 'osmid', 'oneway', 'lanes', 'name', 'highway', 'maxspeed',
       'reversed', 'length', 'bridge', 'width',
       'point', 'snap_dist', 'Transport_number']]

edges_df = final_transport_df[['line_i', 'osmid', 'oneway', 'lanes', 'name', 'highway', 'maxspeed',
       'reversed', 'length', 'bridge', 'geometry', 'width',
       'snap_dist', 'Transport_number']]

In [ ]:
stops_df = stops_df.astype({"osmid": str, "lanes": str, "name": str, "reversed": str, "maxspeed": str, "highway": str})
stops_df["geometry"] = stops_df["point"]
stops_df = gpd.GeoDataFrame(stops_df)
stops_df = stops_df.drop(columns=["point"])

stops_df.set_geometry(col='geometry', inplace=True)


In [ ]:
edges_df = edges_df.astype({"osmid": str, "lanes": str, "name": str, "reversed": str, "maxspeed": str, "highway": str})

In [ ]:
stops_df.to_file("Остановки_Петроградский1_район.geojson", driver="GeoJSON")

In [ ]:
edges_df.to_file("Маршруты_Петроградский1.geojson", driver="GeoJSON")

In [ ]:
route = gpd.read_file('Маршруты_петроградский_район.geojson')
stop = gpd.read_file('Остановки_петроградский_район.geojson')

In [ ]:
route1 = route[route['Transport_number']!= '40']
route1 = route1[route1['Transport_number']!= '1']
stop1 = stop[stop['Transport_number']!= '40']
stop1 = stop1[stop1['Transport_number']!= '1']
stop1 = stop1.astype({"osmid": str, "lanes": str, "name": str, "reversed": str, "maxspeed": str, "highway": str})
route1 = route1.astype({"osmid": str, "lanes": str, "name": str, "reversed": str, "maxspeed": str, "highway": str})

In [ ]:
route1.to_file("Маршруты_Петроградский2.geojson", driver="GeoJSON")
stop1.to_file("Остановки_Петроградский2.geojson", driver="GeoJSON")

In [ ]:
# определяю зону 5-минутной доступности
import requests 

point = (30.290029, 59.962268)    # сюда вводить координаты и адрес точки притяжения
address = 'Аптека Столички'   # 2 точка 59.962268, 30.290029  Аптека Столички
key = 'your key'
transit = '/walking/'
times = [5]

url = ''.join(['https://api.mapbox.com/isochrone/v1/mapbox', 
               transit, str(point[0]), ',', str(point[1]), '?', 
              'contours_minutes=5', '&',
              'contours_colors=6706ce', '&',
               'polygons=true', '&', 
               'access_token=your key',             
              ])

req = requests.get(url)

features = req.json()['features']   # json с данными о маршруте

In [ ]:
isochrones = [Polygon(c['geometry']['coordinates'][0]) for c in features]  # достаю полигон

In [ ]:
F_map = gpd.GeoDataFrame(index = range(len(isochrones)), crs='epsg:4326', geometry = isochrones)  # превращаю зону изохроны 

In [ ]:
near_stops = gpd.sjoin(stop, F_map).drop_duplicates(subset = ['osmid'])   # определяю ближайшие остановки с удалением дубликатов

In [ ]:
near_route_stops = route.copy()[route.copy()['Transport_number'].isin(near_stops['Transport_number'].unique().tolist())].append(
                    stop.copy()[stop.copy()['Transport_number'].isin(near_stops['Transport_number'].unique().tolist())])
# создаю гдф с ребрами и остановками, по которым проезжает транспорт с нужными номерами

In [ ]:
near_route_stops = near_route_stops[near_route_stops['Transport_number']!= '40']
near_route_stops = near_route_stops[near_route_stops['Transport_number']!= '1']

In [ ]:
near_route_stops = near_route_stops.dropna(subset=['name', 'geometry'])

In [ ]:
near_route_stops['street'] = [c[0] if len(c) == 1 else ', '.join(c) for c in near_route_stops['name']]  
# отдельный столбец с названиями улиц, тк ошибка при визуализации из-за списков в name

In [ ]:
f_map = gpd.GeoDataFrame(index = [0], crs='epsg:4326', geometry = [Point(point)])

In [ ]:
f_map['street'] = address

In [ ]:
import folium   # позволяет создать интерактивную карту с несколькими слоями (наподобие ax и plt)

m = near_route_stops.explore(column = 'street', tiles = "CartoDB positron")

f_map.explore(
     m=m,
     color="red",
     marker_kwds=dict(radius=10, fill=True), # make marker radius 10px with fill
)

folium.LayerControl().add_to(m)  # use folium to add layer control

m  # show map

In [ ]:
near_route_stops.explore()

## 3 часть

In [ ]:
# определяю зону 10-минутной доступности
import requests 

point = (30.290029, 59.962268)    # сюда вводить координаты и адрес точки притяжения
address = 'Аптека Столички'   # 2 точка 59.962268, 30.290029  Аптека Столички
key = 'your key'
transit = '/walking/'
times = [10]

url = ''.join(['https://api.mapbox.com/isochrone/v1/mapbox', 
               transit, str(point[0]), ',', str(point[1]), '?', 
              'contours_minutes=5', '&',
              'contours_colors=6706ce', '&',
               'polygons=true', '&', 
               'access_token=your key',             
              ])

req = requests.get(url)

features = req.json()['features']   # json с данными о маршруте

In [ ]:
buildings = gpd.GeoDataFrame(gpd.read_file('houses.geojson'))
routes = gpd.GeoDataFrame(gpd.read_file('houses.geojson'))

In [ ]:
isochrones = [Polygon(c['geometry']['coordinates'][0]) for c in features]  # достаю полигон
isochron = gpd.GeoDataFrame(index = range(len(isochrones)), crs='epsg:4326', geometry = isochrones)  # превращаю зону изохроны 

In [ ]:
buildings['ten_min1'] = buildings.apply(lambda x: isochron.contains(x.geometry), axis = 1)
# buildings['ten_min'] = buildings.apply(lambda x: 1 if x.ten_min1 == True else 0, axis = 1)

In [ ]:
buildings.explore(column = 'ten_min1', cmap = 'Set2', tiles = 'CartoDB positron')

In [ ]:
buildings_over_10 = buildings[buildings['ten_min1']==False]

In [ ]:
buildings_over_10.explore()

In [ ]:
from geojson import Feature, FeatureCollection, dump
from scipy import spatial
import json

with open('pedestrian_1.geojson', encoding = 'utf-8', errors='ignore') as json_file:
    data = json.load(json_file)['features']

G = nx.DiGraph()
for edge in data:
    coord= edge['geometry']['coordinates']
    distance = edge['properties']['distance']
    G.add_edge((coord[0][0], coord[0][1]), (coord[1][0], coord[1][1]), weight = distance)
del edge, coord, distance, data

Tree_Graph = spatial.KDTree(list(G.nodes()))

In [ ]:
gpd.GeoDataFrame(geometry = [Point(c) for c in list(G.nodes())], crs = 4326).explore()

In [ ]:
# Tree_Graph = spatial.KDTree([(c.x, c.y) for c in nodes['geometry']])

In [ ]:
def dist(p0, p1):   # функция рассчёта в расстояния 
    dist = p0.distance(p1)
    return dist

In [ ]:
def search_nearest_point(Tree_Graph, point):
    sub = Tree_Graph.query([point], 1)
    id_node = sub[1].tolist()[0]
    closest_node = Point(Tree_Graph.data[id_node][0], Tree_Graph.data[id_node][1])
    return closest_node

In [ ]:
Tree_Graph = spatial.KDTree(list(G.nodes()))

In [ ]:
keys = [(a.centroid, search_nearest_point(Tree_Graph, (a.centroid.x, a.centroid.y))) for a in buildings_over_10['geometry']]
values = [(a,b) for (a,b) in zip(transport_by_district['ID остановки'], transport_by_district['geometry'])]
all_dist = {k: sorted([[c[0], dist(Point(k[1]),c[1]), search_nearest_point(Tree_Graph, (c[1].x, c[1].y))] for c in values], key=lambda item: item[1])[:5] for k in keys}  
# тут некоторые остановки
all_dist

In [ ]:
for h in all_dist.keys():
    for p in all_dist.values():
        for d in p:
            path = nx.dijkstra_path(G, h[1], d[2], weight = 'weight')
            distance = 0
            for i in range(len(path) - 1):
                a,b = Point(path[i]), Point(path[i+1])
                distance += a.distance(b) 
            d.append(path, distance)

In [ ]:
transport_by_district[transport_by_district['ID остановки']==19099]

In [ ]:
transport_by_district[transport_by_district['ID остановки']==19099].explore()

In [ ]:
buildings_over_10.head()

In [ ]:
P = ox.graph_from_place(TERITORY_NAME, network_type="walk")
ox.plot_graph(ox.project_graph(P))
m1 = ox.plot_graph_folium(P, popup_attribute="name", weight=2, color="#8b0000")

In [ ]:
nodes, edges = ox.utils_graph.graph_to_gdfs(P)
edges.head()

In [ ]:
m = territory.explore(tiles="CartoDB positron")
edges.explore(m=m, marker_kwds={ "color": "red"})